# CAIQ Procurement Eval — End-to-End Pipeline

Run the **complete compliance evaluation loop** from dataset loading through CEP generation,
verdict scoring, and compliance summary — all from one notebook.

**How to use:** Edit the parameters in **Phase 1 (Configuration)**, then run cells one by one.

| Phase | What it does |
|---|---|
| 1 — Configuration | All tunable parameters in one place |
| 2 — Environment | Check API keys and imports |
| 2.5 — Langfuse Health Check | Verify Langfuse credentials before running |
| 3 — Load Dataset | Parse vendor CAIQ xlsx files from int-dataset |
| 4 — Instantiate Agents | Build Planner, Evaluator, Verifier, CCM Lookup |
| 5 — Run Pipeline | Generate CEPs (Plan → Evaluate → Verify) |
| 6 — Inspect First CEP | Browse the first compliance evaluation result |
| 7 — Pass 1: Evaluate Outputs | Compliance verdicts + optional LLM judge |
| 8 — Summarize | Compliance rate by vendor × CCM domain → xlsx + csv |
| 9 — What's Next | Next steps and companion notebook |

## Phase 1 — Configuration

Change these values. Everything else runs automatically.

In [ ]:
import sys
import uuid
from pathlib import Path


# ── Tune these before running ────────────────────────────────────────────────
DATASET_DIR = "../../../int-dataset"   # path to the int-dataset folder
CCM_FILE    = "../../../int-dataset/CCMv4.1.0-generated_at_2026_01_13.xlsx"  # CCM control context
VENDORS     = None                     # None = all vendors; or e.g. ["Google", "Azure", "IBM"]
N_PER_VENDOR = 20                      # max questions per vendor file (keep <=20 for first run)
CONFIG       = "gemini_gemini"         # gemini_gemini | openai_openai | gemini_openai
USE_VERIFIER = True                    # False = skip VerifierAgent (--no_verifier)
USE_CCM      = True                    # False = skip CCM control context lookup
USE_JUDGE    = False                   # True = LLM rubric judge (extra API calls per sample)
N_WORKERS    = 1                       # set >1 to parallelise
# ─────────────────────────────────────────────────────────────────────────────

REPO_ROOT  = Path(".").resolve()
SRC_PATH   = str(REPO_ROOT / "src")
RUN_ID     = str(uuid.uuid4())
OUT_DIR    = str(REPO_ROOT / "ceps" / CONFIG)
OUT_LABEL  = f"{CONFIG}_n{N_PER_VENDOR}"

print(f"Config        : {CONFIG}")
print(f"Dataset dir   : {DATASET_DIR}")
print(f"CCM file      : {CCM_FILE}")
print(f"Vendors       : {VENDORS or 'all'}")
print(f"N per vendor  : {N_PER_VENDOR}")
print(f"Verifier      : {'enabled' if USE_VERIFIER else 'disabled'}")
print(f"CCM lookup    : {'enabled' if USE_CCM else 'disabled'}")
print(f"LLM judge     : {'enabled' if USE_JUDGE else 'disabled'}")
print(f"Run ID        : {RUN_ID}")
print(f"Output dir    : {OUT_DIR}")

## Phase 2 — Environment

Verify API keys and import all modules. Fix any errors here before proceeding.

In [ ]:
import json
import os

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from dotenv import load_dotenv

# Load .env from repo root (two levels up from this notebook)
_env_path = REPO_ROOT.parents[1] / ".env"
if not _env_path.exists():
    _env_path = REPO_ROOT / ".env"
load_dotenv(_env_path)
print(f"Loaded .env from: {_env_path}")

# API key check
missing = []
for var, needed_for in [("GEMINI_API_KEY", "gemini"), ("OPENAI_API_KEY", "openai")]:
    val = os.environ.get(var, "")
    needed = needed_for in CONFIG
    if val and not val.startswith("your_"):
        print(f"  ok      {var}  ({val[:8]}...)")
    elif needed:
        print(f"  MISSING {var}  <- required for {CONFIG}")
        missing.append(var)
    else:
        print(f"  skip    {var}  (not needed for {CONFIG})")

if missing:
    raise EnvironmentError(f"Set {missing} in .env and re-run this cell.")

import pandas as pd
from caiq_procurement_eval.agents.planner_agent import PlannerAgent
from caiq_procurement_eval.agents.evaluator_agent import EvaluatorAgent
from caiq_procurement_eval.agents.verifier_agent import VerifierAgent
from caiq_procurement_eval.datasets.caiq_loader import load_caiq_directory
from caiq_procurement_eval.datasets.caiq_sample import VendorAnswer
from caiq_procurement_eval.cep.writer import iter_ceps
from caiq_procurement_eval.eval.eval_outputs import evaluate_cep_dir
from caiq_procurement_eval.eval.summarize import summarize
from caiq_procurement_eval.langfuse_integration.client import get_client
from caiq_procurement_eval.runner.run_generate_ceps import BACKEND_CONFIGS, process_sample
from caiq_procurement_eval.tools.ccm_lookup_tool import CCMLookupTool

print("\nAll imports OK")

## Phase 2.5 — Langfuse Health Check

Verifies that Langfuse credentials are configured before the pipeline runs.
Langfuse is **optional** — the pipeline produces identical CEPs with or without it.

| Check | What it tests |
|---|---|
| Env vars present | `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` are set in `.env` |
| Client init | `Langfuse()` initialises without error |

In [ ]:
from caiq_procurement_eval.langfuse_integration.client import reset_client

reset_client()

lf_public = os.environ.get("LANGFUSE_PUBLIC_KEY", "")
lf_secret = os.environ.get("LANGFUSE_SECRET_KEY", "")

if not lf_public or not lf_secret:
    print("[skip] LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY are not set.")
    print("       Langfuse tracing is disabled. Pipeline will run fine without it.")
    print()
    print("To enable, add to .env:")
    print("  LANGFUSE_PUBLIC_KEY=pk-lf-...")
    print("  LANGFUSE_SECRET_KEY=sk-lf-...")
else:
    results = {}
    results["env"] = ("ok", f"pk={lf_public[:6]}...")
    try:
        _lf_hc = get_client()
        results["client"] = ("ok", "Langfuse() ready") if _lf_hc else ("fail", "returned None")
    except Exception as e:
        results["client"] = ("fail", str(e))

    all_ok = True
    for key, label in [("env", "Env vars present"), ("client", "Client init     ")]:
        status, detail = results.get(key, ("skip", ""))
        marker = "ok  " if status == "ok" else ("skip" if status == "skip" else "FAIL")
        if status not in ("ok", "skip"):
            all_ok = False
        print(f"  {marker}  {label}  {detail}")

    print()
    if all_ok:
        lf_host = os.environ.get("LANGFUSE_HOST") or "https://cloud.langfuse.com"
        print(f"Langfuse is configured.  Host: {lf_host}")
        print("Traces and scores will be recorded automatically during the pipeline run.")
    else:
        print("WARNING: Langfuse client failed to initialise. Tracing will be skipped.")

## Phase 3 — Load Dataset

Parse all vendor CAIQ xlsx files from `int-dataset`.
Each row becomes a `CAIQSample` — one per CAIQ question-answer pair.

In [ ]:
from collections import Counter

print(f"Loading CAIQ dataset from: {DATASET_DIR}")
print()

samples = load_caiq_directory(
    directory=DATASET_DIR,
    vendors=VENDORS,
    n_per_vendor=N_PER_VENDOR,
)

print(f"\nTotal samples loaded : {len(samples)}")

# Breakdown by vendor
vendor_counts = Counter(s.vendor for s in samples)
print("\nSamples per vendor:")
for vendor, n in sorted(vendor_counts.items()):
    print(f"  {vendor:<40} {n}")

# Breakdown by answer type
answer_counts = Counter(s.vendor_answer.value for s in samples)
print("\nAnswer distribution:")
for ans, n in sorted(answer_counts.items()):
    pct = n / len(samples) * 100
    print(f"  {ans:<20} {n:>4}  ({pct:.1f}%)")

# Breakdown by CCM domain
domain_counts = Counter(s.domain_id for s in samples)
print("\nTop CCM domains:")
for domain, n in sorted(domain_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {domain:<10} {n}")

# Peek at first sample
s0 = samples[0]
print(f"\nFirst sample: {s0.sample_id}")
print(f"  Vendor   : {s0.vendor}")
print(f"  Q ID     : {s0.question_id}  ({s0.control_id})")
print(f"  Domain   : {s0.domain_id}")
print(f"  Answer   : {s0.vendor_answer.value}")
print(f"  Evidence : {s0.vendor_comment[:120]}..." if len(s0.vendor_comment) > 120 else f"  Evidence : {s0.vendor_comment}")

## Phase 4 — Instantiate Agents

Build the pipeline components. Agents are created once and reused across all samples.
CrewAI creates a fresh `Crew` per `run()` call, so this is thread-safe for parallel workers.

| Agent | Role |
|---|---|
| PlannerAgent | Maps CAIQ question → CCM control → evaluation strategy (text-only) |
| EvaluatorAgent | Reads vendor evidence → compliance verdict |
| VerifierAgent | Pass 2.5 — independently cross-checks the verdict |
| CCMLookupTool | Fetches authoritative CCM control text for grounding context |

In [ ]:
config = dict(BACKEND_CONFIGS[CONFIG])
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# PlannerAgent: text-only, derives evaluation strategy
planner = PlannerAgent(
    backend=config["planner_backend"],
    model=config["planner_model"],
)
print(f"PlannerAgent   : {config['planner_backend']} / {config['planner_model']}")

# EvaluatorAgent: reads vendor evidence, produces verdict
evaluator = EvaluatorAgent(
    backend=config["evaluator_backend"],
    model=config["evaluator_model"],
)
print(f"EvaluatorAgent : {config['evaluator_backend']} / {config['evaluator_model']}")

# VerifierAgent: optional Pass 2.5 cross-check
verifier = None
if USE_VERIFIER:
    verifier = VerifierAgent(
        backend=config["evaluator_backend"],
        model=config["evaluator_model"],
    )
    print(f"VerifierAgent  : {config['evaluator_backend']} / {config['evaluator_model']}")
else:
    print("VerifierAgent  : disabled (USE_VERIFIER=False)")

# CCMLookupTool: optional control context grounding
ccm_tool = None
if USE_CCM and CCM_FILE and Path(CCM_FILE).exists():
    ccm_tool = CCMLookupTool(ccm_file=CCM_FILE)
    loaded = ccm_tool.is_loaded()
    print(f"CCMLookupTool  : {'loaded' if loaded else 'failed to load'} ({CCM_FILE})")
elif USE_CCM:
    print(f"CCMLookupTool  : skipped — CCM file not found at {CCM_FILE}")
else:
    print("CCMLookupTool  : disabled (USE_CCM=False)")

# Langfuse observability (no-op if keys not set)
lf_client = get_client()
print(f"Langfuse       : {'enabled' if lf_client else 'not configured'}")

## Phase 5 — Run Pipeline

For each CAIQ sample: **Plan → [CCM Lookup] → Evaluate → [Verify] → write CEP**.

Each call to `process_sample()` is fully self-contained and serialises the result
to `OUT_DIR/<sample_id>.json` as a portable Compliance Evaluation Packet (CEP).

Increase `N_WORKERS` in Phase 1 to parallelise across samples.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


cep_paths = []
print(f"Running {len(samples)} samples  workers={N_WORKERS}  →  {OUT_DIR}\n")


def _run_one(sample):
    return process_sample(
        sample,
        planner,
        evaluator,
        config,
        RUN_ID,
        OUT_DIR,
        lf_client=lf_client,
        verifier=verifier,
        ccm_tool=ccm_tool,
    )


if N_WORKERS <= 1:
    for i, sample in enumerate(samples, 1):
        print(f"[{i}/{len(samples)}] {sample.sample_id} ...", end=" ", flush=True)
        try:
            path = _run_one(sample)
            cep_paths.append(path)
            print("OK")
        except Exception as exc:
            print(f"ERROR: {exc}")
else:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(_run_one, s): s for s in samples}
        for done, fut in enumerate(as_completed(futures), 1):
            s = futures[fut]
            try:
                path = fut.result()
                cep_paths.append(path)
                print(f"[{done}/{len(samples)}] {s.sample_id} → OK")
            except Exception as exc:
                print(f"[{done}/{len(samples)}] {s.sample_id} ERROR: {exc}")

print(f"\nDone — {len(cep_paths)}/{len(samples)} CEPs written to {OUT_DIR}")

## Phase 6 — Inspect First CEP

CEPs are self-contained JSON files. Every field you see here is what the agent actually
produced — no post-processing. The full trace is:

```
Vendor Answer  →  Planner Strategy  →  CCM Context  →  Evaluator Verdict  →  Verifier Check
```

The `lf_trace_id` links this CEP back to the live trace in Langfuse if configured.

In [ ]:
if not cep_paths:
    print("No CEPs written — check errors in Phase 5.")
else:
    cep = json.loads(Path(sorted(cep_paths)[0]).read_text(encoding="utf-8"))
    s   = cep["sample"]
    pl  = cep.get("plan", {}).get("parsed", {})
    ev  = cep.get("evaluation", {}).get("parsed", {})
    vr  = (cep.get("verifier") or {}).get("parsed", {})
    ts  = cep.get("timestamps", {})

    print("=" * 70)
    print(f"Sample ID      : {s['sample_id']}")
    print(f"Vendor         : {s['vendor']}")
    print(f"Question ID    : {s['question_id']}  ({s['control_id']})")
    print(f"Domain         : {s['domain_id']}")
    print(f"CAIQ Version   : {s['caiq_version']}")
    print()
    print(f"Question       : {s['question_text'][:120]}")
    print(f"Vendor Answer  : {s['vendor_answer']}")
    print(f"Vendor Evidence: {s['vendor_comment'][:200]}")
    print()
    print("Planner Strategy:")
    for j, step in enumerate(pl.get("steps", []), 1):
        print(f"  {j}. {step}")
    print(f"  Risk Level     : {pl.get('compliance_risk_level', '--')}")
    print(f"  NIST Mapping   : {pl.get('nist_mapping_hint', '--')}")
    print()
    print(f"Evaluator Verdict    : {ev.get('verdict', '--')}")
    print(f"Confidence           : {ev.get('confidence', '--')}")
    print(f"Evidence Quality     : {ev.get('evidence_quality', '--')}")
    print(f"Evidence Citation    : {ev.get('evidence_citation', '--')[:150]}")
    print(f"Gap Description      : {ev.get('gap_description', '--')}")
    print(f"Reasoning            : {ev.get('reasoning', '--')}")
    if vr:
        print()
        print(f"Verifier Verdict     : {vr.get('verdict', '--')}")
        print(f"Final Verdict        : {vr.get('final_verdict', '--')}")
        print(f"Verifier Reasoning   : {vr.get('reasoning', '--')}")
    print()
    print("Timestamps (ms):")
    for k in ["planner_ms", "evaluator_ms", "verifier_ms"]:
        print(f"  {k:<20} {ts.get(k, 0):.0f}")
    if cep.get("lf_trace_id"):
        print(f"Langfuse trace ID    : {cep['lf_trace_id']}")
    if cep.get("errors"):
        print(f"Errors               : {cep['errors']}")
    print("=" * 70)

## Phase 7 — Pass 1: Evaluate Outputs

Score every CEP for:
- **compliance_score**: compliant=1.0, partial=0.5, non_compliant=0.0 (NA excluded)
- **verifier_verdict**: confirmed / revised / skipped
- **evidence_quality**: strong / adequate / weak / absent
- **judge_*** (optional): 5 rubric dimensions scored 0–1 by LLM judge

Results written to `output/metrics_<label>.jsonl`.

In [ ]:
metrics_path = str(REPO_ROOT / "output" / f"metrics_{OUT_LABEL}.jsonl")
Path(REPO_ROOT / "output").mkdir(parents=True, exist_ok=True)

print(f"Evaluating CEPs in {OUT_DIR} ...\n")

evaluate_cep_dir(
    cep_dir=OUT_DIR,
    out_file=metrics_path,
    no_judge=not USE_JUDGE,
    judge_backend=config.get("judge_backend", "gemini"),
    judge_model=config.get("evaluator_model", "gemini-2.5-flash-lite"),
)

# Load and display quick summary
rows = [json.loads(line) for line in Path(metrics_path).read_text(encoding="utf-8").splitlines() if line.strip()]
df = pd.DataFrame(rows)

print(f"\n--- Quick Summary ({len(df)} samples) ---")

# Verdict distribution
print("\nFinal verdict distribution:")
print(df["final_verdict"].value_counts().to_string())

# Compliance rate by vendor
scored = df[df["compliance_score"].notna()].copy()
scored["compliance_score"] = scored["compliance_score"].astype(float)
if not scored.empty:
    print("\nCompliance rate by vendor:")
    vendor_rates = scored.groupby("vendor")["compliance_score"].mean().sort_values(ascending=False)
    for vendor, rate in vendor_rates.items():
        print(f"  {vendor:<40} {rate:.1%}")

# Evidence quality distribution
print("\nEvidence quality distribution:")
print(df["evidence_quality"].value_counts().to_string())

# Verifier impact
if "verifier_verdict" in df.columns and not df["verifier_verdict"].eq("skipped").all():
    revised = (df["verifier_verdict"] == "revised").sum()
    print(f"\nVerifier revised : {revised}/{len(df)} ({revised / len(df):.1%})")

## Phase 8 — Summarize

Aggregate metrics into a compliance report by vendor × CCM domain.
Writes both a CSV and an Excel workbook with multiple sheets.

**Trace chain visualised:**
```
CAIQ Question (AIS-01.1)
    ↓ maps to
CCM Control (AIS-01)  ←  CCMLookupTool fetches control specification
    ↓ evaluated against
Vendor Evidence ("We follow ISO 27001...")
    ↓ produces
Compliance Verdict (compliant, confidence=0.9)
    ↓ with
Evidence Citation (exact quote from vendor comment)
    ↓ packaged into
CEP (fully auditable JSON trace)
```

In [ ]:
summary_path = str(REPO_ROOT / "output" / f"summary_{OUT_LABEL}.csv")

summarize(metrics_file=metrics_path, out_file=summary_path)

# Load and display
summary_df = pd.read_csv(summary_path)
print("\n--- Compliance Rate by Vendor × CCM Domain ---")
print(summary_df.to_string(index=False))

# Gap hotspots: domains with lowest compliance
if not summary_df.empty:
    print("\n--- Top 5 Compliance Gaps (lowest scoring domains) ---")
    gap_df = summary_df.sort_values("compliance_rate").head(5)
    for _, row in gap_df.iterrows():
        print(f"  {row['vendor']:<30} {row['domain_id']:<8} {row['compliance_rate']:.1%}  (n={int(row['n_questions'])})")

## Phase 9 — What's Next

- **`analysis.ipynb`** — visualise the results: compliance charts, verifier revision analysis,
  gap breakdown by vendor and domain, judge score distributions, per-sample trace browser

- **Compare vendors** — adjust `VENDORS` in Phase 1 to focus on specific providers

- **Compare configs** — change `CONFIG` to `openai_openai` or `gemini_openai` and re-run
  Phases 4–8 to compare model performance on the same dataset

- **Enable judge** — set `USE_JUDGE=True` for 5-rubric interpretability scoring per CEP

- **Increase scale** — raise `N_PER_VENDOR` and `N_WORKERS` for a full run

- **Langfuse dashboard** — if configured, every verdict trace is browsable with
  full planner/evaluator/verifier spans, token counts, and latency at
  [cloud.langfuse.com](https://cloud.langfuse.com)

In [ ]:
print("Pipeline complete.")
print(f"  CEPs      : {OUT_DIR}")
print(f"  Metrics   : {metrics_path}")
print(f"  Summary   : {summary_path}")
if lf_client:
    lf_host = os.environ.get("LANGFUSE_HOST") or "https://cloud.langfuse.com"
    print(f"  Langfuse  : {lf_host}")